In [1]:
import json
import pandas as pd

In [4]:
import gzip, json

DATASET_PATH = "Pfam38.0/dataset.clean.jsonl.gz"

def stream_dataset(path):
    with gzip.open(path, "rt") as f:
        for line in f:
            yield json.loads(line)

# quick sanity check
for i, rec in zip(range(2), stream_dataset(DATASET_PATH)):
    print(rec["protein_id"], len(rec["sequence"]), rec["domains"])


Q8TWI7 470 [{'pfam_id': 'PF21799', 'start': 2, 'end': 97}]
Q8TWN7 435 [{'pfam_id': 'PF21799', 'start': 9, 'end': 96}]


In [5]:
rec

{'protein_id': 'Q8TWN7',
 'sequence': 'MSVDELPFDSVAVIGLGVEGRTCAARLAELGLKVYASDIREDVDVSSLERLPNVEVELGHHDIDRIRECDTVYFSPSIPNDAEIVREVRNIGVPPLEDLLTWPDSSKFVVVSGTNGKTTTVHMIDHLLDKLGIDRKIGGNAGGGFDGYATLYVRAELARPEVVVCEVCDMTLDYFSDRAPRYDVAVFLNLGLDHLDYHGSIERYAARAARFCDRARETAIVKCDGEERKVVERMESTPRCYDELDVEVDIPLRGEFYRLDAKAALLAVAEITSEPPEELAPLLSDFTSVPGRIKEYKLGEGRLVIGKTDNLSALKVVLRDYGPLDVVFWGTPRPGEDYRIEGIGRTLREAGVERVYVFPGLSEETVDDVIEELTSSGIEASPVEPSEVASKALELARRGRDVGVLGNGQDVIISIQHDVERAVRFLTAGSERSRG',
 'domains': [{'pfam_id': 'PF21799', 'start': 9, 'end': 96}]}

In [6]:
from collections import Counter

pfam_counter = Counter()

for rec in stream_dataset(DATASET_PATH):
    for d in rec["domains"]:
        pfam_counter[d["pfam_id"]] += 1

pfam_ids = sorted(pfam_counter)
pfam2idx = {pfam: i+1 for i, pfam in enumerate(pfam_ids)}
idx2pfam = {i: pfam for pfam, i in pfam2idx.items()}

NUM_LABELS = len(pfam2idx) + 1  # +0 = background
print("Number of Pfam labels:", NUM_LABELS)


Number of Pfam labels: 10


In [7]:
pfam_ids

['PF00001',
 'PF01225',
 'PF02603',
 'PF04613',
 'PF07085',
 'PF10309',
 'PF14111',
 'PF17306',
 'PF21799']

In [8]:
import numpy as np

def build_residue_labels(seq_len, domains, pfam2idx):
    """
    Returns: np.ndarray of shape (seq_len,)
    """
    labels = np.zeros(seq_len, dtype=np.int64)  # background = 0

    for d in domains:
        pf = d["pfam_id"]
        if pf not in pfam2idx:
            continue
        s = d["start"] - 1   # to 0-based
        e = d["end"]         # end is inclusive in Pfam
        labels[s:e] = pfam2idx[pf]

    return labels


In [9]:
rec = next(stream_dataset(DATASET_PATH))
y = build_residue_labels(len(rec["sequence"]), rec["domains"], pfam2idx)

print("Sequence length:", len(y))
print("Unique labels:", set(y))


Sequence length: 470
Unique labels: {0, 9}


In [10]:
rec

{'protein_id': 'Q8TWI7',
 'sequence': 'MKRVIVVGAGSAGRSVARLLNHVGYDVVINDIRDWEDFTTEEREYLEVLEREGVEVALGGHDRELFESADAAFVSPAIPEDAEGRKLAELVPEEITVEDVAGLLGDLFPVPAIGITGTNGKTTTTWAVAHLLEACGYEVWRCSSLDRHLVIEAIVDGVVTGEIEGYDVAVVELPHGTIRLAAGIELDVGVLTNVSPEHLDEFGGSFERYVRRKLTITEMSEVLVAGYDCEELRKRLNDAVWYSVRDPDADYYGRRENGRLVVKGPDASVEADFHLIGYYVENCTGAVAACLEFGADPEGVAEGLATFEGVPGRMELLGEFSGRVAYIDAAHNPDGLKASLPPFRELADERGGRLLVSVDNPDTVTERDKFEFGRIVGEYADFVAASGYNETLERLDRSAAEEVVKGAESAGCEGVAVDSVREAAEEVAARSEEGDVLLHVGPGVVNAYDRVKRAFLRGLTSVLGEPSDEA',
 'domains': [{'pfam_id': 'PF21799', 'start': 2, 'end': 97}]}